# Logistic Regression Baseline

In [13]:
from pathlib import Path
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

In [14]:
DATA_DIR = Path("data/merged")

train_df = pd.read_csv(DATA_DIR / "FPS_train.csv")
val_df = pd.read_csv(DATA_DIR / "FPS_val.csv")
test_df = pd.read_csv(DATA_DIR / "FPS_test.csv")

In [15]:
# define a logistic regression model
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
    )),
    ("classifier", LogisticRegression(
        C=1.2,
        max_iter=1000,
    ))
])

# Train without automatically rendering the fitted pipeline in Jupyter.
_ = model.fit(
    train_df["review"],
    train_df["voted_up"]
)

In [16]:
# evaluate model performance
def evaluate_model(model, data):
    y_true = data["voted_up"]
    y_pred = model.predict(data["review"])

    print(classification_report(
        y_true, y_pred, labels=[1,0], target_names=["recommended", "not recommend"],
        zero_division=0, digits=3
    ))

    print("Macro-F1:", f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0,
    ))
    
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[1,0]))

In [17]:
evaluate_model(model, val_df)

               precision    recall  f1-score   support

  recommended      0.886     0.734     0.803       308
not recommend      0.824     0.930     0.873       412

     accuracy                          0.846       720
    macro avg      0.855     0.832     0.838       720
 weighted avg      0.850     0.846     0.843       720

Macro-F1: 0.8381370366844827
Confusion matrix:
[[226  82]
 [ 29 383]]


In [18]:
print("Test set results:")
evaluate_model(model, test_df)

Test set results:
               precision    recall  f1-score   support

  recommended      0.862     0.688     0.765       308
not recommend      0.798     0.918     0.854       413

     accuracy                          0.820       721
    macro avg      0.830     0.803     0.809       721
 weighted avg      0.825     0.820     0.816       721

Macro-F1: 0.8094732819462062
Confusion matrix:
[[212  96]
 [ 34 379]]
